In [1]:
# Importing the Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler,LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import BaggingRegressor

# load the dataset
data = pd.read_excel('BBDM Project for 2nd research article.xlsx')

# Columns to exclude
columns_to_exclude = ['Chainage','Formation','RMC', ]

# Preprocessing: Exclude specified columns
X = data.drop(['PRnet'] + columns_to_exclude, axis=1)
Y = data['PRnet']

# Label encoding for multiple columns
label_encoder = LabelEncoder()
for col in ['Lithology', 'Weathering', 'Rock Strength']:
    X[col] = label_encoder.fit_transform(X[col])

# Train Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)  # Removed 'stratify' as it is not used in regression

# Data Standardization
scaler = RobustScaler()
scaler.fit(X_train)  # Fit on the training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the BaggingRegressor
model = BaggingRegressor()

# Define the grid of hyperparameters
param_grid = {
    'n_estimators': list(range(10, 301, 10)),
    'max_samples': np.round(np.arange(0.1, 1.1, 0.1), 2),
    'max_features': np.round(np.arange(0.1, 1.1, 0.1), 2),
    'bootstrap': [True, False],
    'bootstrap_features': [True, False],
}


grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5,
                           scoring='neg_mean_absolute_error',
                           n_jobs=-1, verbose=2)
                           
grid_result = grid_search.fit(X_train_scaled, Y_train)

# Print the best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)

# Evaluate the model
best_regressor = grid_search.best_estimator_
Y_pred = best_regressor.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("Mean Absolute Error:", mae)
print("R^2 Score:", r2)

Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best Hyperparameters: {'bootstrap': True, 'bootstrap_features': False, 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 100}
Mean Absolute Error: 2.3336834203907917
R^2 Score: 0.9363687310895713
